# LITE Re-Ranker — Kaggle Training Notebook

This notebook drives the `literank` package (a faithful reference implementation of the
**LITE** document re-ranker, [arXiv:2406.17968](https://arxiv.org/abs/2406.17968)) on
Kaggle's free GPU tier.

**Before running:**

1. In the Kaggle notebook sidebar, set **Accelerator -> GPU T4 x2** and turn **Internet -> On**
   (needed to `pip install`, pull the HuggingFace encoder/teacher models, and download MS MARCO).
2. Kaggle GPU sessions are capped (currently ~30 GPU-hours/week, ~12h/session, often less on a
   shared T4x2). Training to the paper's `max_steps=20000` will usually **span multiple
   sessions**. This notebook checkpoints every `checkpoint_every` steps
   (see `literank.config.TrainConfig`) under `/kaggle/working/ckpt_*`, and the `train` CLI
   command accepts `--resume <path_to_ckpt.pt>` to continue from the last saved step. The final
   markdown cell below explains how to persist `/kaggle/working/ckpt_*` as a Kaggle Dataset so
   the next session can resume training instead of starting over.
3. This is a **qualitative reproduction**, not a leaderboard run: the paper reports MRR@10 =
   0.393 on the full MS MARCO dev set with full-scale training compute. With Kaggle's free GPU
   and a `--subset-size` slice of MS MARCO, expect numbers below 0.393 — the goal is to see the
   LITE scorer learn and to demonstrate the ablations (LITE vs MaxSim, Small-LITE projection
   storage, activation choice), not to match the paper's headline number.


In [ ]:
# Clone/locate the repo and install the package in editable mode.
# On Kaggle, either attach this repo as a Dataset/utility-script, or clone it (Internet must be ON).
%cd /kaggle/working
!test -d searchandrank || git clone https://github.com/<your-org>/searchandrank.git
%cd /kaggle/working/searchandrank

!pip install -q -e .
# Fallback if the editable install above fails in the Kaggle sandbox:
# !pip install -q torch transformers datasets scikit-learn


In [ ]:
# Offline smoke test: fast unit tests only, no network/model downloads.
!uv run pytest -m "not integration" -q
# If uv is not available on the Kaggle image, fall back to plain pytest:
# !pytest -m "not integration" -q


In [ ]:
# Train the LITE scorer on a subset of MS MARCO, distilled from the cross-encoder teacher.
# Checkpoints land in /kaggle/working/ckpt_lite/ckpt_step<N>.pt every `checkpoint_every` steps.
!python -m literank.cli train --scorer lite --proj-dim 768 \
    --subset-size 100000 --max-steps 20000 \
    --checkpoint-dir /kaggle/working/ckpt_lite --device cuda


In [ ]:
# Train the MaxSim (ColBERT-style) baseline for the LITE-vs-MaxSim ablation.
!python -m literank.cli train --scorer maxsim --subset-size 100000 \
    --max-steps 20000 --checkpoint-dir /kaggle/working/ckpt_maxsim --device cuda


In [ ]:
# Small-LITE storage ablation: compare cached embedding bytes at proj_dim=768 vs proj_dim=128.
# `encode_and_cache` returns the size (bytes) of the saved cache file -- the "storage lever"
# from the paper's Small-LITE variant (projecting to a smaller d' shrinks the offline doc cache).
import torch
from literank.config import ModelConfig
from literank.encoder import DualEncoder

sample_docs = [
    "The quick brown fox jumps over the lazy dog.",
    "LITE is a lightweight late-interaction re-ranker for document retrieval.",
    "Kaggle provides free GPU compute for training and inference.",
] * 20  # small stand-in batch; the dev-doc cache built below uses the real MS MARCO dev split

from literank.encode_cache import encode_and_cache

cfg_768 = ModelConfig(proj_dim=768)
enc_768 = DualEncoder(cfg_768).to("cuda")
bytes_768 = encode_and_cache(enc_768, sample_docs, cfg_768.doc_len,
                              "/kaggle/working/ablation_proj768.pt", batch_size=32)

cfg_128 = ModelConfig(proj_dim=128)
enc_128 = DualEncoder(cfg_128).to("cuda")
bytes_128 = encode_and_cache(enc_128, sample_docs, cfg_128.doc_len,
                              "/kaggle/working/ablation_proj128.pt", batch_size=32)

print(f"proj_dim=768 cache size: {bytes_768:,} bytes")
print(f"proj_dim=128 cache size: {bytes_128:,} bytes")
print(f"storage ratio (128/768): {bytes_128 / bytes_768:.3f}")
assert bytes_128 < bytes_768, "Small-LITE (proj_dim=128) should use less storage than proj_dim=768"


In [ ]:
# Encode a dev slice of MS MARCO, cache document embeddings, rerank candidates per
# query against the trained LITE checkpoint, and compute MRR@10 / nDCG@10 with literank.evaluate.
from datasets import load_dataset
from literank.config import ModelConfig, DataConfig
from literank.encoder import DualEncoder
from literank.model import Ranker
from literank.checkpoint import load_checkpoint
from literank.encode_cache import encode_and_cache, load_embeddings
from literank.rerank import rerank
from literank.evaluate import mrr_at_k, ndcg_at_k

dcfg = DataConfig()
mcfg = ModelConfig(scorer="lite", proj_dim=768)

dev = load_dataset(dcfg.dataset_name, dcfg.dataset_config, split="validation")
dev = dev.select(range(min(dcfg.num_dev_queries, len(dev))))

# Load the trained LITE checkpoint produced by the training cell above.
ranker = Ranker(mcfg).to("cuda")
import glob, os, re

def latest_checkpoint(ckpt_dir):
    # Checkpoints are written every `checkpoint_every` steps and a Kaggle session is
    # often interrupted before `max_steps`, so pick the highest-numbered checkpoint
    # rather than hard-coding a step count.
    paths = glob.glob(os.path.join(ckpt_dir, "ckpt_step*.pt"))
    if not paths:
        raise FileNotFoundError(f"no checkpoints in {ckpt_dir}")
    return max(paths, key=lambda p: int(re.search(r"ckpt_step(\d+)\.pt", os.path.basename(p)).group(1)))

latest_ckpt = latest_checkpoint("/kaggle/working/ckpt_lite")
load_checkpoint(latest_ckpt, ranker, map_location="cuda")
ranker.eval()

all_relevances = []
for rec in dev:
    query = rec["query"]
    passages = rec["passages"]["passage_text"]
    is_selected = rec["passages"]["is_selected"]
    if not passages:
        continue

    # Cache this query's candidate passages, then rerank with the trained scorer.
    doc_cache_path = "/kaggle/working/dev_doc_cache.pt"
    encode_and_cache(ranker.encoder, passages, mcfg.doc_len, doc_cache_path, batch_size=32)
    doc_embs, doc_masks = load_embeddings(doc_cache_path)

    q_emb, q_mask = ranker.encoder.encode([query], mcfg.query_len)
    order = rerank(ranker.scorer, q_emb, q_mask, doc_embs, doc_masks, device="cuda")

    relevances = [is_selected[i] for i in order]
    all_relevances.append(relevances)

mrr10 = mrr_at_k(all_relevances, k=10)
ndcg10 = ndcg_at_k(all_relevances, k=10)
print(f"Dev MRR@10:  {mrr10:.4f}")
print(f"Dev nDCG@10: {ndcg10:.4f}")
print("Note: the paper reports MRR@10=0.393 on the full MS MARCO dev set at full training "
      "scale. This subset/Kaggle run is expected to score lower -- it is a qualitative "
      "reproduction of the LITE architecture and training recipe, not a leaderboard result.")


## Multi-session training: checkpoint -> Kaggle Dataset -> `--resume`

Kaggle GPU sessions are time-boxed, so a full `--max-steps 20000` run will often need to
continue across multiple sessions. Use the checkpoint files this notebook already writes to
`/kaggle/working/ckpt_lite/` and `/kaggle/working/ckpt_maxsim/` (one `ckpt_step<N>.pt` file per
`checkpoint_every` steps, per `literank.config.TrainConfig`):

1. **At the end of a session**, open the notebook's **Output** tab (or the right-hand "Data"
   pane), and click **"New Dataset"** from the `/kaggle/working/ckpt_lite` (or `ckpt_maxsim`)
   folder. Kaggle will snapshot those checkpoint files as a private Dataset you can re-attach
   to a new session.
   - Alternatively, save the whole notebook ("Save Version") with "Always save output" enabled,
     which persists `/kaggle/working/*` as the notebook's output and lets you create a Dataset
     from it afterward.
2. **In the next session**, add that Dataset as an input (Notebook -> Add Data -> Your
   Datasets), which mounts it read-only under `/kaggle/input/<dataset-name>/`.
3. Copy (or symlink) the most recent `ckpt_step<N>.pt` somewhere writable, then resume.
   Because a session can be interrupted at any step, `<N>` varies with wherever the
   previous session stopped -- it is **not** a fixed value like `ckpt_step20000.pt`.
   Find the checkpoint with the highest `<N>` first, e.g. with the same helper the
   training/eval cells above use:

```python
import glob, os, re

def latest_checkpoint(ckpt_dir):
    paths = glob.glob(os.path.join(ckpt_dir, "ckpt_step*.pt"))
    if not paths:
        raise FileNotFoundError(f"no checkpoints in {ckpt_dir}")
    return max(paths, key=lambda p: int(re.search(r"ckpt_step(\d+)\.pt", os.path.basename(p)).group(1)))

latest_ckpt = latest_checkpoint("/kaggle/input/<dataset-name>")
```

Then copy that file and resume:

```python
!cp <latest_ckpt_path_from_above> /kaggle/working/resume_ckpt.pt
!python -m literank.cli train --scorer lite --proj-dim 768 \
    --subset-size 100000 --max-steps 40000 \
    --checkpoint-dir /kaggle/working/ckpt_lite \
    --resume /kaggle/working/resume_ckpt.pt --device cuda
```

`train --resume <path>` calls `literank.checkpoint.load_checkpoint`, which restores the model,
optimizer, and AMP grad-scaler state and returns the saved `step`, so training continues from
exactly where it left off (raise `--max-steps` past the resumed step to keep going). Repeat the
save-as-Dataset / re-attach / `--resume` loop each session until training reaches the desired
`max_steps`.
